# Q_1

**1. How can we improve face detection by adding reidentification to anonymize only specific individuals while keeping computational cost low?**

El principal reto de añadir reidentificación facial es conseguir identificar de forma fiable a una misma persona a pesar de cambios de postura, ángulo de la cámara, iluminación, oclusiones, etc. 

Una arquitectura eficiente tendría que separar el problema en tres etapas:
1. Detección de caras utilizando el detector YOLO existente para localizar las caras en los frames. 
2. Tracking para mantener el seguimiento de cada cara a lo largo de los frames para evitar ejecutar el modelo de reconocimiento facial en cada frame. Aquí se puede utilizar ByteTrack que es de bajo coste y proporciona un identificador temporal (track_id). 
3. Reidentificación cuando sea necesario, és decir, si aparece un nuevo track_id, se pierde el tracking o periódicamente para verificar la identidad, se ejecutaría un modelo de reconocimiento como por ejemplo uno basado en MobileNet entrenado con una función de pérdida como ArcFace. Este modelo generaría un embedding de la cara. 

Posteriormente, el embedding se compararia mediante cosine similarity con los embeddings de las personas que queremos identificar. Si la similitud supera un umbral determinado, se puede considerar que se trata de una identidad conocida y aplicar la anonimización correspondiente. 

La principal ventaja de este sistema es que el coste computacional se reduce al no ejecutar el modelo de reidentificación en todos los frames, ya que serían el detector y el tracker los que funcionarian continuamente, mientras que el reidentificador se ejecutaria de forma esporádica. 

También sería útil usar un sistema de caching de embeddings asociado al track_id, de forma que una vez identificada una persona se pueda reutilizar el resultado mientras el tracking sea fiable. 

Además, en caso de una identidad ambigua, es preferible anonimizar antes que dejar una persona identificable, ya que en este caso un falso negativo es más crítico que un falso positivo. 

**2. How can we extend an anonymization pipeline with depth-based filtering to blur only background objects efficiently?**

Para incorporar información de profundidad al pipeline de anonimización primero haría una estimación de profundidad y después decidiria sobre que regiones deben anonimizarse. 

Una primera aproximación sería utilizar un modelo de estimación de profundidad. En el caso de video, no lo ejecutaría para cada frame ni a resolución completa, de forma que se reduciría el coste computacional. Se podría:
1. Reducir la resolución del frame
2. Estimar el mapa de profundidad
3. Hacer upsampling del mapa obtenido a la resolución original
4. Actualizar la estimación para N frames e interpolarla entre frames

Para decidir las zonas que hay que anonimizar no aplicaría un threshold directamente sobre la profundidad, ya que la interpretación depende de la escena. Lo que haría sería combinarlo con detección o segmentación. \
Si el objetivo es anonimizar determinados objetos del fondo, se puede utilizar el mapa de profundidad para identificar las regiones que están detrás de una determinada distancia. Seguidamente aplicaría un detector/segmentador para determinar exactamente qué objetos pertenecen a las regiones del fondo. Por ejemplo, podría darse el caso que un objeto quedase cortado al aplicar el threshold, por lo tanto, se tendría que detectar el objeto completo si este está contenido total o parcialmente en el background. \
Además, para video incorporaría tracking de objetos. De esta forma el detector se ejecutaría solo cuando fuera necesario y el tracker mantendría las máscaras entre los frames. 

Otra posibilidad es si el objetivo es simplemente separar foreground y background. En este caso la solución sería utilizar un modelo de matting, para video una opción sería RVM. Esto puede ser más adecuado si la decisión depende principalmente de separar a la persona del fondo. 

**3. Imagine we have this case: our face detection system does not detect properly (bad accuracy) dark-skins nor bald-people; how would you improve system accuracy?**

Lo primero que haría sería determinar si realmente existe un problema de sesgo del modelo y localizar exactamente dónde se produce. 

No utilizaría únicamente métricas globales como mAP, ya que una métrica agregada puede ocultar rendimientos significativamente peor en determinados subgrupos. Construiría un conjunto de evaluación estratificado por atributos relevantes y mediría:
* Recall
* Precision
* mAP
* Rendimiento según condiciones de iluminación
* Rendimiento según resolución, pose y oclusión

En este caso el recall es importante, ya que un falso negativo significa que una cara no ha sido detectada y, por lo tanto, podría quedar sin anonimizar. 

También analizaría ejemplos concretos de errores para determinar la causa. Por ejemplo:
* Si las personas de piel oscura fallan en determinadas condiciones de iluminación
* Si las personas calvas fallan porque el modelo ha aprendido características relacionadas con el contorno del pelo
* Si los errores aparecen en determinados ángulos o resoluciones

El principal problema en estos casos puede estar en la distribución de características de estos ejemplos, ya que al estar poco representada en los datos de entrenamiento el modelo no generaliza bien. 

Para solucionarlo probaría varias opciones. 
* Datos: aumentar la diversidad del dataset incorporando diferentes tonos de piel, tipos de pelo, personas calvas, iluminación, poses y resolución. 
* Data augmentation: aplicar transformaciones realistas de iluminación, constraste, blur, ruido, escalado, oclusión, etc. 
* Entrenamiento: utilizar estrategias como focal loss o reweighting si se observa que el modelo no aprende correctamente las clases minoritarias. También probaría fine-tuning o una arquitectura con más capacidad si el modelo actual es demasiado pequeño. 
* Evaluación: tener un conjunto de test estratificado que permita comprobar si hay una mejora. 

**4. Imagine this other case: let's assume we have added several bald people imagery to our dataset and we still are not increasing the accuracy. What may be happening? How would you fix this?**

Si al añadir más ejemplos de personas calvas no mejora el rendimiento, puede ser que el principal problema no venga del volumen de datos. Primero identificaría en que parte del pipeline está el problema. 

Estas podrían ser las causas del problema:
1. Datos\
Puede ser que aunque se hayan añadido más datos la distribución de personas calvas siga siendo insuficiente. Además, de no tener suficiente variabilidad en condiciones de iluminación, resolución o postura. \
En este caso el modelo puede aprender esos ejemplos concretos, pero no generalizar a personas calvas en diferentes condiciones. \
También comprobaría posibles duplicados o fuga de información entre train y validation/test, ya que si imagenes similares de una misma persona aparecen en varios conjuntos las métricas pueden ser poco representativas. 
2. Anotaciones \
Revisaría manualmente las anotaciones de los nuevos datos, ya que errores en las bounding boxes pueden dificultar el aprendizaje, sobre todo si las cajas son inconsistentes entre anotadores.
3. Entrenamiento \
Es posible que el modelo sea demasiado pequeño para capturar las caractrísticas necesarias, aunque en ese caso primero intentaría descartar problemas de datos y anotaciones. \
En caso de hacer fine-tuning de un modelo preentrenado, comprobaría si el backbone está congelado o si el learning rate es demasiado bajo para que el modelo pueda adaptarse correctamente. \
También compararía experimentos con diferentes estrategias de entrenamiento y, si es necesario, probaría una arquitectura un poco mayor. \
También se podrían utilizar técnicas como focal loss o reweighting si se observa que los ejemplos difíciles reciben poca importancia durante el entrenamiento. 
4. Evaluación \
Comprobaría que el conjunto de test tenga suficientes ejemplos representativos de personas calvas y que no estemos evaluando únicamente una métrica global. \
Crearía un conjunto de evaluación específico para este caso y mediría recall, precision y tasa de falsos negativos.

Además, realizaría experimentos controlados cambiando una sola variable cada vez y compararía los resultados en el mismo conjunto de evaluación. De esta forma se puede saber si realmente la mejora viene de los nuevos datos, de la función de pérdida o de los cambios en el modelo. 

**5. Which MLOPs infrastructure would you design to manage this system? Just provide a high-level design and which tools would you use.**

Diseñaría la infraestructura que permita versionar los datos, automatizar el entrenamiento, comprobar la calidad antes de desplegarlo, monitorizarlo y reentrenarlo. Además, al estar sujeto al AI Act y RGPD debe permitir trazabilidad y supervisión humana. 

El sistema lo dividiría en dos partes:
* Por un lado la parte de inferencia (edge), es decir, donde se ejecutaría el modelo en tiempo real y se anonimizan las imágenes.
* Por otro lado estaría la parte de entrenamiento (en la nube o en servidores propios). Aquí se almacenan y versionan los datos, se entrenan y evalúan los modelos y se monitoriza el sistema. 

Componentes:
1. Gestión de datos\
Los datos y los conjuntos de entrenamiento se tienen que versionar para poder reporducir cualquier experimento. Para ello, usaría DVC que se puede integrar con Git. Además, los datos los almacenaría en un object storage como S3.
2. Entrenamiento y tracking de experimentos\
Se puede automatizar el entrenamiento en un pipeline con Airflow siguiendo los siguientes pasos: preparación de los datos, entrenamiento, evaluación y exportación del modelo. \
Para registrar hiperparámetros, datos y resultados correspondientes a cada experimento usaría MLflow.
3. Evaluación\
Antes de pasar a producción cada modelo debe superar unos criterios automáticos (mediante GitHub CI) sobre un conjunto de test fijo como: 
    * recall global y por subgrupos
    * tasa de falsos negativos
    * latencia y FPS
    * comparación con el modelo que ya está en producción
4. Monitorización\
Una vez está en producción es necesario controlar latencia, uso de GPU, errores. Para ello usaría herramientas como Prometheus que recopila y almacena métricas de rendimiento, y Grafana que sirve como herramienta de visualización. 

Además, definiría un mecanismo de feedback desde producción para recopilar fallos, estos se podrían revisar e incorporar al dataset de forma controlada para futuros nuevos experimentos. 

**6. From end user POV, which techniques would you use to improve final solution user experience?**

Desde el punto de vista del usuario final, el sistema de anonimización de video debería ser estable, rápido y fiable. 
1. Estabilidad temporal\
Uno de los problemas más visibles para el usuario sería el flickering, es decir, cuando una cara aparece anonimizada durante un frame y deja de estarlo en el siguiente. \
En este caso utilizaría tracking, como ByteTrack, para mantener la identidad temporal del objeto y propagar la máscara de anonimizaciñon durante varios frames. En este caso usaría un filtro de Kalman para predecir la posición cuando temporalmente se pierde la detección. 
2. Margen en la bounding box\
Ampliaría la región de la bounding box y utilizaría un borde suavizado para evitar que parte de la cara sea visible, ya que es preferible cierto exceso a dejar una parte identificable. 
3. Baja latencia\
Optimizaría el modelo y el pipeline para mantener una latencia baja y un FPS estable. 
4. Personalización\
Proporcionaría opción de configurar aspectos como la intensidad del blur, pixelación o margen de seguridad, aunque con limitaciones siguiendo las políticas de privacidad. 
5. Gestión de errores\
Incluiria un mecanismo para reportar casos en los que una persona no ha sido anonimizada, una región ha sido anonimizada incorrectamente, se ha producido flickering, etc. \
Este feedback podría servir para mejorar el modelo en fases posteriores. 

**7. (EXTRA) Provide a code snippet (practical example) on how would you implement any of these applications: reidentification or depth estimation, for face anonymization.**

Reidentificación

In [ ]:
import cv2
import nummpy as np 

# Modelo de face recognition que genera embeddings. 
# Podría ser un modelo basado en MobileNet + ArcFace.
face_recognizer = load_face_recognition_model()

# Embeddings de referencia de las personas que queremos identificar.
known_embeddings = {
    "person_1": load_embedding("person_1.npy"),
    "person_2": load_embedding("person_2.npy")
}

track_identity = {}

COSINE_THRESHOLD = 0.5


def cosine_similarity(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return np.dot(a, b)

def identify_face(face_image):
    embedding = face_recognizer(face_image)

    best_identity = None 
    best_similarity = -1

    for identity, reference_embedding in known_embeddings.item():
        similarity = cosine_similarity(
            embedding,
            reference_embedding
        )

        if similarity > best_similarity:
            best_similarity = similarity
            best_identity = identity

    if best_similarity >= COSINE_THRESHOLD:
        return best_identity
    
    return None


for frame in vide_stream:

    # Detección de caras
    detections = face_detection(frame)

    # Tracking
    tracks = tracker.update(detections)

    for track in tracks:
        track_id = track.id 
        x1, y1, x2, y2 = track.box 

        # Reconocimiento cuando hay un nuevo track.
        if track_id not in track_identity:
            face = frame[y1:y2, x1:x2]
            identity = identify_face(face)
            track_identity[track_id] = identity
        else: 
            identity = track_identity[track_id]
        
        # Aplicar anonimización según la identidad
        if identity in known_embeddings.keys():
            face = frame[y1:y2, x1:x2]
            blurred_face = cv2.GaussianBlur(
                face, 
                (31, 31),
                0
            )

            frame[y1:y2, x1:x2] = blurred_face

La implementación está muy simplificada. Es un caso real se podrían hacer las siguientes mejoras:
* Alineamiento facial antes de generar el embedding
* Utilizar varios embeddings de referencia por persona para representar diferentes poses y condiciones
* Ajustar el threshold sobre un conjunto de validación
* Reidentificación periódica para evitar errores persistentes del tracker
* Eliminar el estado asociado a un track_id cuando desaparece el track
* Tratamiento conservador en los casos de baja confianza, priorizando la privacidad

Además, otra mejora que implementaría en un sistema con un número elevado de identidades sería sustituir el diccionario de embeddings por una base de datos vectorial. En lugar de comparar el embedding generado con todas las referencias, se podría almacenar en una solución como Qdrant o PostgreSQL con pgvector y realizar una búsqueda de los vecinos más similares. \
Esto permitiría que la solución escalase mejor a medida que aumentase el número de identidades y facilitaría la gestión de altas, bajas y actualización de los embeddings. 